# Week 19, Day 2: Posterior summaries, prediction and Bayes factors

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w019/d2.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

### Exercise: Prediction with uncertainty about the rate

With a Gamma(2, 0.5) prior and counts [3, 7, 4] from 3 trials, find the posterior, then draw 200,000 predictive counts by first drawing a rate from the posterior and then a Poisson count with that rate. Compare the predictive variance with the plug-in Poisson variance.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from math import lgamma
rng = np.random.default_rng(1995)
k = np.array([3, 7, 4])
post_a, post_b = ..., ...
rates = rng.gamma(post_a, 1 / post_b, 200000)
pred = ...
print(f"posterior Gamma({post_a}, {post_b}); predictive mean {pred.mean():.2f}, variance {pred.var():.2f};  plug-in Poisson variance {post_a / post_b:.2f}")

In [ ]:
import numpy as np
def test_predictive():
    "the predictive samples have a mean close to the posterior mean rate"
    assert abs(pred.mean() - post_a / post_b) < 0.1

def test_wider():
    "the predictive variance exceeds the plug-in Poisson variance (the posterior mean) by at least 20%"
    assert pred.var() > 1.2 * (post_a / post_b)

test_predictive()
test_wider()
print("All checks passed")

### Exercise: Is this neuron Poisson?

A neuron's 60 trial counts come from a Poisson process with a fluctuating gain (over-dispersed). Fit the gamma-Poisson model for a single rate with a Gamma(1, 0.1) prior. For 2,000 replicates, draw a rate from the posterior, simulate 60 Poisson counts, and record their Fano factor. Compute the fraction of replicates with a Fano factor at least as large as the observed one.

In [ ]:
counts = rng.poisson(8 * rng.gamma(4, 1 / 4, 60))          # the real data: over-dispersed
fano_obs = counts.var(ddof=1) / counts.mean()
pa, pb = 1 + counts.sum(), 0.1 + len(counts)
fano_rep = []
for _ in range(2000):
    lam = ...
    rep = ...
    fano_rep.append(rep.var(ddof=1) / rep.mean())
fano_rep = np.array(fano_rep)
ppc_p = ...
print(f"observed Fano factor {fano_obs:.2f};  replicated Fano factors mostly between {np.percentile(fano_rep, 2.5):.2f} and {np.percentile(fano_rep, 97.5):.2f};  predictive p = {ppc_p:.4f}")
plt.hist(fano_rep, 40); plt.axvline(fano_obs, color="k"); plt.xlabel("Fano factor"); plt.show()

In [ ]:
import numpy as np
def test_reps():
    "fano_rep holds one Fano factor per replicated dataset"
    assert fano_rep.shape == (2000,)

def test_misfit():
    "the observed Fano factor is beyond almost all replicates (posterior predictive p below 0.01)"
    assert ppc_p < 0.01

test_reps()
test_misfit()
print("All checks passed")

### Exercise: Lindley's paradox

Write `log_ev_fair(k, n)` = log P(k | p = 0.5) using `lgamma`, and `log_ev_flexible(n)` = log(1/(n + 1)), the evidence when p is uniform. For k = 5,100 and n = 10,000, compute the Bayes factor in favour of the fair model, `bf01`, and the two-sided p-value of the z-test, `p_value`.

In [ ]:
from math import erf, sqrt, log, exp
def log_choose(n, k): return lgamma(n + 1) - lgamma(k + 1) - lgamma(n - k + 1)

def log_ev_fair(k, n):
    return ...

def log_ev_flexible(n):
    return ...

k_b, n_b = 5100, 10000
bf01 = exp(log_ev_fair(k_b, n_b) - log_ev_flexible(n_b))
z = (k_b / n_b - 0.5) / sqrt(0.25 / n_b)
p_value = 2 * (1 - 0.5 * (1 + erf(abs(z) / sqrt(2))))
print(f"z = {z:.2f}, p = {p_value:.4f};  Bayes factor for p = 0.5 against unknown p: {bf01:.1f}")

In [ ]:
import numpy as np
def test_evidence():
    "log_ev_flexible equals log(1 / (n + 1)), the evidence under a uniform prior"
    assert abs(log_ev_flexible(10) - np.log(1 / 11)) < 1e-12

def test_paradox():
    "for 5,100 bursts in 10,000 trials, p is below 0.05 but the Bayes factor favours p = 0.5 by more than 5 to 1"
    assert p_value < 0.05 and bf01 > 5

test_evidence()
test_paradox()
print("All checks passed")